In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote
from pathlib import Path

AUDIO_EXTENSIONS = {".mp3", ".wav", ".flac", ".aac", ".ogg", ".m4a"}

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def is_audio(file_name):
    return any(file_name.lower().endswith(ext) for ext in AUDIO_EXTENSIONS)

def get_soup(url):
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

def clean_name(name):
    return unquote(name).split("?")[0]

def download_file(url, output_path):
    print(f"Downloading: {output_path}")
    r = requests.get(url, headers=HEADERS)
    r.raise_for_status()

    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with open(output_path, "wb") as f:
        f.write(r.content)

def crawl_folder(url, local_path):
    soup = get_soup(url)

    for a in soup.find_all("a"):
        href = a.get("href")
        if not href:
            continue

        full_url = urljoin(url, href)
        text = clean_name(a.text.strip())

        # Skip parent navigation links
        if text in ("", "..", "Parent folder"):
            continue

        # Folder detection (Dropbox uses folder links without dl=1)
        if "dl=0" in full_url and not is_audio(text):
            # Could be folder OR file preview; try treating as folder first
            if "/folders/" in full_url or "." not in text:
                new_local_path = os.path.join(local_path, text)
                crawl_folder(full_url, new_local_path)
                continue

        # Convert to direct download
        download_url = full_url.replace("dl=0", "dl=1")

        if is_audio(text):
            file_path = os.path.join(local_path, text)
            download_file(download_url, file_path)

def main():
    root_url = "https://www.dropbox.com/scl/fo/p1xamlw94wt1p29de542e/h?dl=0"

    cwd = Path.cwd()
    workspace_root = cwd
    for parent in [cwd, *cwd.parents]:
        if (parent / "Dataset").exists():
            workspace_root = parent
            break

    output_dir = workspace_root / "Dataset"
    output_dir.mkdir(parents=True, exist_ok=True)

    crawl_folder(root_url, str(output_dir))

if __name__ == "__main__":
    main()